# Project 07: Burrows-Wheeler Transform

**Note**: Keep the cell below as is. It is for formatting purposes

In [1]:
%%html
<style>
.MathJax_Display,  /* MathJax 2 */
mjx-container {    /* MathJax 3 */
  text-align: left !important;
}
</style>

## Learning Objectives

1. Students will be able to explain the fundamental principles of the Burrows-Wheeler Transform algorithm, including its purpose, process, and applications in data compression and bioinformatics.
2. Students will be able to implement the Burrows-Wheeler Transform using both the naive approach and the more efficient suffix array method.
3. Students will be able to construct auxiliary data structures (count array and occurrence array) needed for efficient pattern matching with the FM-Index.
4. Students will be able to apply the BWT-based pattern matching algorithm to find exact string matches within a reference text.

---

## Background
The Burrows-Wheeler Transform (BWT) is a powerful algorithm used in data compression and bioinformatics. Developed by Michael Burrows and David Wheeler in 1994, it rearranges a character string into runs of similar characters, making it more amenable to compression techniques like move-to-front encoding and run-length encoding. Despite adding a small amount of overhead, BWT's ability to group similar characters together significantly improves compression efficiency.

The BWT works by sorting all possible rotations of a string lexicographically and then extracting the last column of the sorted matrix. What makes BWT remarkable is that this transformation is reversible without storing much additional information - only the position of the original string in the sorted rotations is needed.

In recent years, BWT has found extensive applications beyond compression. It's particularly valuable in bioinformatics for efficient sequence alignment of DNA reads against reference genomes, as implemented in popular tools like Bowtie and BWA. The algorithm's efficiency in handling large datasets has made it indispensable for processing the massive amounts of data generated by next-generation sequencing technologies.

## The BWT Algorithm

![](assets/rotation.png)

> Credit: Ferragina, P., Giancarlo, R., Manzini, G., & Sciortino, M. (2005). Boosting textual compression in optimal linear time. In Journal of the ACM (Vol. 52, Issue 4, pp. 688–713). Association for Computing Machinery (ACM). https://doi.org/10.1145/1082036.1082043 

At its core, the BWT algorithm involves creating all cyclic rotations of an input string, sorting them lexicographically, and then extracting the last column.

Let's define the pseudocode for our BWT function:

$$
{\small
\begin{array}{l}
\textbf{function } \text{BWT}(text) \\
\quad \text{Append a unique end-of-string character to } text \text{ if not present} \\
\quad \text{Create a list to store all possible rotations of the text} \\
\quad \textbf{for each } \text{possible starting position in the text} \textbf{ do} \\
\quad\quad \text{Generate a rotation starting at this position and add to the list} \\
\quad \text{Sort all rotations lexicographically} \\
\quad \text{Initialize an empty string for the transformed result} \\
\quad \textbf{for each } \text{rotation in the sorted list} \textbf{ do} \\
\quad\quad \text{Append the last character of this rotation to the transformed result} \\
\quad \textbf{return } \text{transformed result and position of original text in sorted rotations}
\end{array}
}
$$

In [2]:
import random


def generate_dna(length: int = 150, seed: int = 42) -> str:
    """Generate a random DNA sequence of the given length.

    Uses a fixed seed so the same sequence comes out every time the
    notebook is run, which keeps the test outputs reproducible.

    Args:
        length: Number of bases to generate.
        seed: Random seed for reproducibility.

    Returns:
        A random string made of A, C, G, and T.
    """
    random.seed(seed)
    return "".join(random.choices("ACGT", k=length))


dna_seq = generate_dna()
print(f"Random DNA sequence ({len(dna_seq)} bp):")
print(dna_seq)

Random DNA sequence (150 bp):
GACAGGTACAAGAAGGAGTATGCATCAATGTGGTCGTGTGGAACAAACGCCACTGGAGACTGGGTTAACCATTCGCTCCAGCGTCATGAAAGTCACTGTTAGGGCGACCTTCGATTCGGATGTGACATTTCATTACATTACGCTCAGGAC


In [3]:
def BWT(string: str) -> str:
    """Function to calculate Burrows-Wheeler Transform for a given string.
    
    Computes the Burrows-Wheeler Transform by creating all rotations of the input
    string, sorting them lexicographically, and extracting the last column.
    
    Args:
        string: The input string to transform.
    
    Returns:
        The Burrows-Wheeler Transform of the input string.
        
    Examples:
        >>> BWT('googol')
        'lo$oogg'
        
        >>> BWT('banana$')
        'annb$aa'
    """
    # '$' may appear at most once, and only at the end
    if string.count('$') > 1 or ('$' in string and not string.endswith('$')):
        raise ValueError("'$' must appear only once, at the end of the string")

     # append $ if it isn't already at the end
    if not string.endswith('$'):
        string = string + '$'

    # empty list for the rotations, start from the original string
    rotations = []
    current = string

    # one pass per character; add first, then rotate
    for char in string:
        rotations.append(current)
        current = current[1:] + current[0]

    # sort lexicographically (in place)
    rotations.sort()

    # last character of each sorted rotation
    bwt = "".join(rotation[-1] for rotation in rotations)
    
    return bwt

# Docstring examples
print("BWT('googol')  :", BWT('googol'), "| expected: lo$oogg")
print("BWT('banana$') :", BWT('banana$'), "| expected: annb$aa")

# Testing the BWT function with the generated DNA sequence

dna_bwt = BWT(dna_seq)
print("\nBWT of random DNA sequence:")
print(dna_bwt)
print("Length:", len(dna_bwt), "| expected:", len(dna_seq) + 1)

BWT('googol')  : lo$oogg | expected: lo$oogg
BWT('banana$') : annb$aa | expected: annb$aa

BWT of random DNA sequence:
CCGGTACGACGATGTGAGATCGCAGCCATCGACCTGACCCGCAAATCTCTAGTTACAGTAAGTAATTGTGGAAACTGAG$TCAGGGCTCGACCTATACGATATGGAACGTTTCGTTGTTTGAGCGTCTTTGAGAGCCGAGACGAAGTCAAA
Length: 151 | expected: 151


While this approach works, it's inefficient for large strings. A more efficient implementation uses suffix arrays.

## Suffix Arrays for BWT

Suffix arrays provide a memory-efficient way to compute the BWT. A suffix array contains the starting positions of all suffixes of a string, sorted lexicographically.

$$
\small
\begin{array}{l}
\textbf{function } \text{suffix\_array}(text) \\
\quad \text{Create a list of all suffixes of the text, each paired with its starting position} \\
\quad \text{Sort these suffix-position pairs lexicographically by suffix} \\
\quad \text{Extract just the starting positions from the sorted pairs} \\
\quad \textbf{return } \text{the list of sorted starting positions}
\end{array}
$$


In [4]:
def suffix_array(string: str) -> list[int]:
    """Function to calculate suffix-array for a given string.
    
    Computes the suffix array by sorting all suffixes of the input string
    lexicographically and returning their starting positions.
    
    Args:
        string: The input string to process.
    
    Returns:
        A list of integers representing the starting positions of the
        lexicographically sorted suffixes.
        
    Examples:
        >>> suffix_array('googol')
        [6, 3, 0, 5, 2, 4, 1]
        
        >>> suffix_array('banana$')
        [6, 5, 3, 1, 0, 4, 2]
    """
    # '$' may appear at most once, and only at the end
    if string.count('$') > 1 or ('$' in string and not string.endswith('$')):
        raise ValueError("'$' must appear only once, at the end of the string")

    # add $ if it isn't already at the end
    if not string.endswith('$'):
        string = string + '$'

    # empty list for (suffix, position) pairs
    pairs = []

    # one pair per starting position
    for i in range(len(string)):
        suffix = string[i:]
        pairs.append((suffix, i))

    # sort by suffix (first element of each pair)
    pairs.sort()

    # keep only the positions, in sorted order
    positions = [pair[1] for pair in pairs]

    return positions

# Docstring examples
print("suffix_array('googol')  :", suffix_array('googol'), "| expected: [6, 3, 0, 5, 2, 4, 1]")
print("suffix_array('banana$') :", suffix_array('banana$'), "| expected: [6, 5, 3, 1, 0, 4, 2]")

# Testing the suffix_array function with the generated DNA sequence
dna_sa = suffix_array(dna_seq)
print("\nFirst 10 entries:", dna_sa[:10])
print("First entry is the $ suffix:", dna_sa[0] == len(dna_seq))


suffix_array('googol')  : [6, 3, 0, 5, 2, 4, 1] | expected: [6, 3, 0, 5, 2, 4, 1]
suffix_array('banana$') : [6, 5, 3, 1, 0, 4, 2] | expected: [6, 5, 3, 1, 0, 4, 2]

First 10 entries: [150, 44, 88, 41, 66, 45, 9, 12, 89, 26]
First entry is the $ suffix: True


Once we have the suffix array, we can compute the BWT more efficiently:

$$
\small
\begin{array}{l}
\textbf{function } \text{BWT\_from\_suffix\_array}(text, \text{suffix\_positions}) \\
\quad \text{Initialize an empty string of the same length as the input text} \\
\quad \textbf{for each } \text{position in the suffix positions list do} \\
\quad\quad \textbf{if } \text{the suffix starts at the beginning of the text then} \\
\quad\quad\quad \text{Add the last character of the text to the transformed result} \\
\quad\quad \textbf{else} \\
\quad\quad\quad \text{Add the character preceding the suffix to the transformed result} \\
\quad \textbf{return } \text{the transformed result}
\end{array}
$$

![](assets/full.png)

> Credit: Siren, J. (2016). Burrows-Wheeler Transform for Terabases. In 2016 Data Compression Conference (DCC) (pp. 211–220). 2016 Data Compression Conference (DCC). IEEE. https://doi.org/10.1109/dcc.2016.17 

In [5]:
def BWT_from_suffix_array(
  text: str, 
  suffix_positions: list[int]
  ) -> str:
    """Function to calculate the Burrows-Wheeler Transform from a suffix array.
    
    Computes the Burrows-Wheeler Transform by using the suffix array to identify
    the character that precedes each suffix in the sorted order.
    
    Args:
        text: The input string to transform.
        suffix_positions: The suffix array for the input string, containing the
            starting positions of all suffixes in lexicographical order.
    
    Returns:
        The Burrows-Wheeler Transform of the input string.
        
    Examples:
        >>> BWT_from_suffix_array("banana$", [6, 5, 3, 1, 0, 4, 2])
        'annb$aa'
        
        >>> BWT_from_suffix_array("googol$", [6, 3, 0, 5, 2, 4, 1])
        'lo$oogg'
    """
    # '$' may appear at most once, and only at the end
    if text.count('$') > 1 or ('$' in text and not text.endswith('$')):
        raise ValueError("'$' must appear only once, at the end of the string")

    # add $ if it isn't already at the end (same as BWT and suffix_array)
    if not text.endswith('$'):
        text = text + '$'
    # empty string to collect one BWT character per slot
    bwt = ""

    # go through the suffix array in the order given (the order of SA is already sorted)
    for p in suffix_positions:

        # find the character right before position p
        if p == 0:
            # nothing before position 0, so wrap around to the last char ($)
            char = text[-1]
        else:
            char = text[p - 1]

        # add it to the end of the result
        bwt += char

    #  return the finished BWT
    return bwt

# Docstring examples
print("BWT_from_suffix_array('banana$') :", BWT_from_suffix_array('banana$', [6, 5, 3, 1, 0, 4, 2]), "| expected: annb$aa")
print("BWT_from_suffix_array('googol$') :", BWT_from_suffix_array('googol$', [6, 3, 0, 5, 2, 4, 1]), "| expected: lo$oogg")

# random DNA sequence, cross-checked against the rotation method
dna_bwt_sa = BWT_from_suffix_array(dna_seq + '$', dna_sa)
print("\nMatches rotation-based BWT:", dna_bwt_sa == dna_bwt)

BWT_from_suffix_array('banana$') : annb$aa | expected: annb$aa
BWT_from_suffix_array('googol$') : lo$oogg | expected: lo$oogg

Matches rotation-based BWT: True


---

## FM-Index for Pattern Matching

The BWT becomes particularly powerful when combined with auxiliary data structures to form an FM-Index, which enables efficient pattern matching. Two key components are the count and occurrence arrays.

The FM-index is a compressed full-text index data structure that combines the Burrows-Wheeler Transform (BWT) with auxiliary data structures to enable efficient pattern matching while maintaining a small memory footprint. Developed by Paolo Ferragina and Giovanni Manzini (hence the name "FM"), it's sometimes referred to as "Full-text Minute-space" index.

#### Core Components

The FM-index consists of several key components:

1. **The BWT (L column)** - The last column from the Burrows-Wheeler Matrix, which is a permutation of the original text that groups similar characters together

2. **First column (F)** - Can be represented as an array of |Σ| integers (where Σ is the alphabet), or sometimes not stored at all as it can be derived

3. **Count array (C)** - Stores the number of characters lexicographically smaller than each character in the alphabet

4. **Occurrence table (OT)** - Tracks how many times each character appears up to each position in the BWT

5. **Suffix Array (SA) sample** - A subset of the suffix array positions, often stored at regular intervals to save space

#### Key Operations

The FM-index supports several fundamental operations:

1. **Count** - Returns the number of occurrences of a pattern in the text in O(p) time, where p is the pattern length

2. **Locate** - Returns the positions of all occurrences in the text

3. **Backward search** - The core algorithm that enables efficient pattern matching by traversing the pattern from right to left

4. **LF mapping** - A procedure that maps positions in the L column to corresponding positions in the F column

#### Advantages

The FM-index offers several notable benefits:

1. **Space efficiency** - Occupies space close to the entropy of the indexed text, typically 5nHk(T) + o(n) bits, where Hk(T) is the k-th order entropy

2. **Query performance** - Allows searching for pattern occurrences in O(p + occ log^(1+ε) n) time, where occ is the number of occurrences

3. **Self-indexing** - Encapsulates the indexed data, allowing reconstruction of the original text without storing it separately

4. **Compression** - Takes advantage of the compressibility of the indexed data

5. **Partial decompression** - Only decompresses tiny portions of the compressed file during queries

The FM-index has become particularly important in bioinformatics for efficient DNA and protein sequence alignment, forming the basis of popular tools like Bowtie and BWA that handle massive genomic datasets.

### The implementation

One of the key parts for string matching is to do Last-to-First column mapping (LF mapping) within the BWT matrix. With the LF property, we  need to build two dictionaries for our reference string beforehand:

1. count: e.g.  `{'A': 0, 'C': 2, 'G': 3, 'T': 5}`

Where for each character `a` in a string, `count[a]` contains the number of characters in string that are lexicographically smaller than `a`.

2. occur: `{'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}`

Where for each character `a` in a bwt string, `occur[a][i]` contains the number of occurences of `a` in `bwt_string[0,i], i=1,...,len(bwt_string)` (i.e. the first i characters in bwt string).

With those two dictionaries, we can then start matching the query string to our reference string. 

## Count Array

The count array stores the number of characters lexicographically smaller than each character in the alphabet:

$$
\small
\begin{array}{l}
\textbf{function } \text{calculate\_counts}(\text{transformed}, \text{alphabet}) \\
\quad \text{Initialize a dictionary to track character counts} \\
\quad \text{Initialize a result dictionary for cumulative counts} \\
\quad \text{Set initial cumulative count to zero} \\
\quad \textbf{for each } \text{character in the sorted alphabet do} \\
\quad\quad \text{Store the current cumulative count for this character} \\
\quad\quad \text{Increase the cumulative count by the frequency of this character} \\
\quad \textbf{return } \text{the dictionary of cumulative counts}
\end{array}
$$

In [6]:
from collections import Counter


def cal_count(string: str) -> dict[str, int]:
    """Function to count characters lexicographically smaller than each character.
    
    For each character in the alphabet, calculates how many characters in the
    input string are lexicographically smaller than it.
    
    Args:
        string: The input string to analyze.
    
    Returns:
        A dictionary mapping each character to the count of characters
        lexicographically smaller than it.
    
    Examples:
        >>> cal_count('ATGACG')
        {'A': 0, 'C': 2, 'G': 3, 'T': 5}
        
        >>> cal_count('banana')
        {'a': 0, 'b': 3, 'n': 4}
    """
    # count how many times each character shows up
    char_counts = Counter(string)

    # running total starts at 0
    running_total = 0

    # empty dictionary for the answers
    count = {}

    #  walk through the characters in alphabetical order
    for char in sorted(char_counts):
        # store the total first (only smaller chars are counted so far)
        count[char] = running_total
        # then add this char's own count for the next letter up
        running_total += char_counts[char]

    # return the finished dictionary
    return count

# Docstring examples
print("cal_count('ATGACG') :", cal_count('ATGACG'), "| expected: {'A': 0, 'C': 2, 'G': 3, 'T': 5}")
print("cal_count('banana') :", cal_count('banana'), "| expected: {'a': 0, 'b': 3, 'n': 4}")

# Testing the cal_count function with the generated DNA sequence
dna_count = cal_count(dna_seq)
print("\nBase counts:", dict(Counter(dna_seq)))
print("cal_count:  ", dna_count)


cal_count('ATGACG') : {'A': 0, 'C': 2, 'G': 3, 'T': 5} | expected: {'A': 0, 'C': 2, 'G': 3, 'T': 5}
cal_count('banana') : {'a': 0, 'b': 3, 'n': 4} | expected: {'a': 0, 'b': 3, 'n': 4}

Base counts: {'G': 40, 'A': 41, 'C': 33, 'T': 36}
cal_count:   {'A': 0, 'C': 41, 'G': 74, 'T': 114}


## Occurrence Array

The occurrence array tracks how many times each character appears up to each position in the BWT:

$$
\small
\begin{array}{l}
\textbf{function } \text{calculate\_occurrences}(\text{transformed}, \text{alphabet}) \\
\quad \text{Initialize a dictionary mapping each character to an array of zeros} \\
\quad \textbf{for each } \text{position in the transformed text do} \\
\quad\quad \text{Identify the character at the current position} \\
\quad\quad \textbf{for each } \text{character in the alphabet do} \\
\quad\quad\quad \text{Copy the previous occurrence count to the current position} \\
\quad\quad \text{Increment the occurrence count for the current character} \\
\quad \textbf{return } \text{the dictionary of occurrence counts}
\end{array}
$$


In [7]:
def cal_occur(bwt_string: str) -> dict[str, list[int]]:
    """Function to calculate occurrences of each character up to each position.
    
    For each character and each position i, calculates how many times the
    character appears in the substring bwt_string[0:i].
    
    Args:
        bwt_string: The BWT string to analyze.
    
    Returns:
        A dictionary mapping each character to a list of occurrence counts,
        where occur[char][i] is the number of occurrences of char in
        bwt_string[0:i].
    
    Examples:
        >>> cal_occur('AG$CG')
        {'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}
        
        >>> cal_occur('annb$aa')
        {'$': [0, 0, 0, 0, 1, 1, 1], 'a': [0, 1, 1, 1, 1, 2, 3], 'b': [0, 0, 0, 1, 1, 1, 1], 'n': [0, 0, 2, 2, 2, 2, 2]}
    """
    # each unique character gets its own list of zeros
    alphabet = sorted(set(bwt_string))
    occur = {}
    for char in alphabet:
        occur[char] = [0] * len(bwt_string)

    # walk the BWT one position at a time
    for i in range(len(bwt_string)):
        current_char = bwt_string[i]

        # copy every tally from the previous position
        # position 0 there's nothing to copy, so it stays 0
        if i > 0:
            for char in alphabet:
                occur[char][i] = occur[char][i - 1]

        # click for the character sitting at this position
        occur[current_char][i] += 1

    # return the finished table
    return occur

# Docstring examples
print("cal_occur('AG$CG')  :", cal_occur('AG$CG'))
print("expected            : {'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}")
print("\ncal_occur('annb$aa'):", cal_occur('annb$aa'))

# Testing the cal_occur function with the generated DNA sequence
dna_occur = cal_occur(dna_bwt)
print("\nLast tally of each row:", {c: dna_occur[c][-1] for c in dna_occur})
print("Base counts in BWT:    ", dict(sorted(Counter(dna_bwt).items())))

cal_occur('AG$CG')  : {'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}
expected            : {'$': [0, 0, 1, 1, 1], 'A': [1, 1, 1, 1, 1], 'C': [0, 0, 0, 1, 1], 'G': [0, 1, 1, 1, 2]}

cal_occur('annb$aa'): {'$': [0, 0, 0, 0, 1, 1, 1], 'a': [1, 1, 1, 1, 1, 2, 3], 'b': [0, 0, 0, 1, 1, 1, 1], 'n': [0, 1, 2, 2, 2, 2, 2]}

Last tally of each row: {'$': 1, 'A': 41, 'C': 33, 'G': 40, 'T': 36}
Base counts in BWT:     {'$': 1, 'A': 41, 'C': 33, 'G': 40, 'T': 36}


---

# Pattern Matching with BWT

![](assets/pattern_match.png)

> Credit: ["DNA Sequence Alignment with the Burrows Wheeler Transform"](https://cs.carleton.edu/cs_comps/2324/sequenceAlignment/) from Carleton College

## Alignments using BWT

With the BWT and auxiliary structures in place, we can efficiently find pattern matches using backward search:

$$
\small
\begin{array}{l}
\textbf{function } \text{update\_range}(\text{character}, \text{range\_start}, \text{range\_end}, \text{counts}, \text{occurrences}) \\
\quad \text{Calculate new start position using the character's count and occurrences at range start} \\
\quad \text{Calculate new end position using the character's count and occurrences at range end} \\
\quad \textbf{return } \text{new start and end positions}
\end{array}
$$

In [8]:
def update_range(
  lower: int, 
  upper: int, 
  count: dict[str, int], 
  occur: dict[str, list[int]], 
  a: str) -> tuple[int, int]:
    """Function to update range given character a.
    
    Updates the search range during backward search in the BWT pattern matching
    algorithm when processing character a.
    
    Args:
        lower: The lower boundary of the current range.
        upper: The upper boundary of the current range.
        count: Dictionary mapping each character to the count of lexicographically
            smaller characters.
        occur: Dictionary mapping each character to its occurrence counts at each
            position.
        a: The character being processed in the pattern.
    
    Returns:
        A tuple containing the updated lower and upper boundaries of the range.
    
    Note:
        This function assumes occur[a][-1] = 0 for boundary conditions.
    """
    # tally of 'a' just before the range starts
    # if lower is 0 there's nothing before it, so use 0, not occur[a][-1] --> missed this initially and things broke down very badly.
    if lower == 0:
        tally_before = 0
    else:
        tally_before = occur[a][lower - 1]

    # tally of 'a' at the end of the range
    tally_end = occur[a][upper]

    # jump to the 'a' block in F (+1 for the '$'), skip the earlier a's
    new_lower = count[a] + 1 + tally_before

    # last 'a' inside our range
    new_upper = count[a] + tally_end

    # return the new range
    return new_lower, new_upper

# hand-traced banana steps: searching 'ana' right to left
b_count = cal_count('banana')
b_occur = cal_occur('annb$aa')
print("rows 0-6, 'a' :", update_range(0, 6, b_count, b_occur, 'a'), "| expected: (1, 3)")
print("rows 1-3, 'n' :", update_range(1, 3, b_count, b_occur, 'n'), "| expected: (5, 6)")
print("rows 5-6, 'a' :", update_range(5, 6, b_count, b_occur, 'a'), "| expected: (2, 3)")

# Testing the update_range function with the generated DNA sequence

full_upper = len(dna_bwt) - 1
for base in "ACGT":
    lo, hi = update_range(0, full_upper, dna_count, dna_occur, base)
    print(f"'{base}' block: rows {lo}-{hi}, size {hi - lo + 1} | expected size: {dna_seq.count(base)}")

rows 0-6, 'a' : (1, 3) | expected: (1, 3)
rows 1-3, 'n' : (5, 6) | expected: (5, 6)
rows 5-6, 'a' : (2, 3) | expected: (2, 3)
'A' block: rows 1-41, size 41 | expected size: 41
'C' block: rows 42-74, size 33 | expected size: 33
'G' block: rows 75-114, size 40 | expected size: 40
'T' block: rows 115-150, size 36 | expected size: 36


<div style="text-align: left">
    
$$
\small
\begin{array}{l}
\textbf{function } \text{find\_match}(\text{pattern}, \text{transformed}, \text{counts}, \text{occurrences}, \text{suffix\_positions}) \\
\quad \text{Initialize search range to cover the entire transformed text} \\
\quad \textbf{for each } \text{character in the pattern, processing from right to left do} \\
\quad\quad \text{Update the search range based on the current character} \\
\quad\quad \textbf{if } \text{the range becomes empty then} \\
\quad\quad\quad \textbf{return } \text{empty list as pattern is not found} \\
\quad \text{Collect all suffix positions within the final range} \\
\quad \textbf{return } \text{the list of matching positions}
\end{array}
$$

</div>

In [9]:
def find_match(query: str, reference: str) -> list[int]:
    """Function to find exact matching by applying Burrows-Wheeler Transform.
    
    Searches for all occurrences of the query string within the reference string
    using the Burrows-Wheeler Transform algorithm for efficient pattern matching.
    
    Args:
        query: The pattern string to search for.
        reference: The text string to search within.
    
    Returns:
        A list of integers representing the 0-based starting positions of all
        occurrences of the query string within the reference string. Returns an
        empty list if no matches are found.
    
    Examples:
        >>> find_match('ana', 'banana')
        [1, 3]
        
        >>> find_match('xyz', 'banana')
        []
    """
    # add $, then build the suffix array and the BWT
    text = reference + '$'
    sa = suffix_array(text)
    bwt = BWT_from_suffix_array(text, sa)

    # count from the plain reference, occur from the BWT
    count = cal_count(reference)
    occur = cal_occur(bwt)

    # start with every row as a possible match
    lower = 0
    upper = len(bwt) - 1

    # read the query from right to left, shrinking the range
    for letter in reversed(query):
        # letter isn't in the reference at all, so no match is possible
        if letter not in count:
            return []

        lower, upper = update_range(lower, upper, count, occur, letter)

        # range is empty, so the query isn't in the reference
        if lower > upper:
            return []

    # turn the remaining rows into positions using the suffix array
    positions = []
    for row in range(lower, upper + 1):
        positions.append(sa[row])

    return sorted(positions)

# Docstring examples
print("find_match('ana', 'banana') :", find_match('ana', 'banana'), "| expected: [1, 3]")
print("find_match('xyz', 'banana') :", find_match('xyz', 'banana'), "| expected: []")
print("find_match('na', 'banana')  :", find_match('na', 'banana'), "| expected: [2, 4]")

# Testing the find_match function with the generated DNA sequence
queries = [dna_seq[10:20], "GAC", "CATT", "TTTT", "NNN"]
for q in queries:
    bwt_hits = find_match(q, dna_seq)
    print(f"{q:>10}: {bwt_hits}")

find_match('ana', 'banana') : [1, 3] | expected: [1, 3]
find_match('xyz', 'banana') : [] | expected: []
find_match('na', 'banana')  : [2, 4] | expected: [2, 4]
AGAAGGAGTA: [10]
       GAC: [0, 57, 105, 123, 147]
      CATT: [69, 125, 130, 135]
      TTTT: []
       NNN: []


***

## Run-Length Encoding

The Burrows-Wheeler Transform tends to group identical characters into runs, which makes it a natural candidate for run-length encoding (RLE). In RLE, we store each run as a pair: the character and the length of its consecutive occurrence. This is particularly effective when the transformed text contains many long runs.

Conceptually, run-length encoding on the BWT string works as follows:

$$
\small
\begin{array}{l}
\textbf{function } \text{run\_length\_encode}(\text{input text}) \\
\quad \text{Start with an empty encoded text} \\
\quad \textbf{if } \text{the input text is empty then} \\
\quad\quad \textbf{return } \text{the empty encoded text} \\
\quad \text{Remember the first character of the input text} \\
\quad \text{Set a counter to one for how many times this character appears in a row} \\
\quad \textbf{for each } \text{next character in the input text} \textbf{ do} \\
\quad\quad \textbf{if } \text{the character is the same as the one being counted} \\
\quad\quad\quad \text{Increase the counter by one} \\
\quad\quad \textbf{else} \\
\quad\quad\quad \text{Add the character being counted and its counter to the encoded text} \\
\quad\quad\quad \text{Begin counting from one for the new character} \\
\quad \text{After the scan, add the final character and its counter to the encoded text} \\
\quad \textbf{return } \text{the encoded text}
\end{array}
$$


In [10]:
def run_length_encode(bwt_string: str) -> str:
    """Run-length encode a BWT string as a compact character-count string.

    Scans the input BWT string from left to right and groups consecutive
    identical characters into runs, returning an encoded string where each
    run is represented as the character followed by its count.

    Args:
    bwt_string: The Burrows-Wheeler transformed string to encode.

    Returns:
    A run-length encoded string of the form 'a3n2b1$1a2', where each
    character is followed by the length of its consecutive run.

    Examples:
    >>> run_length_encode('aaabbc')
    'a3b2c1'

    >>> run_length_encode('annb$aa')
    'a1n2b1$1a2'
    """
     # nothing to encode
    if bwt_string == "":
        return ""

    # start tracking the first character
    encoded = ""
    current_char = bwt_string[0]
    current_count = 1

    # go through the rest of the string
    for char in bwt_string[1:]:
        if char == current_char:
            # same letter, the run keeps going
            current_count += 1
        else:
            # different letter, so the run ended: write it out
            if current_count > 1:
                encoded += f"{current_count}{current_char}"
            else:
                encoded += current_char
            # start tracking the new letter (already seen once)
            current_char = char
            current_count = 1

    # write out the last run, which never hit the else above
    if current_count > 1:
        encoded += f"{current_count}{current_char}"
    else:
        encoded += current_char

    return encoded

# Docstring examples
print("run_length_encode('aaabbc')  :", run_length_encode('aaabbc'), "| expected: 3a2bc")
print("run_length_encode('annb$aa') :", run_length_encode('annb$aa'), "| expected: a2nb$2a")
print("run_length_encode('')        :", repr(run_length_encode('')), "| expected: ''")

# Testing the run_length_encode function with the generated DNA sequence
dna_rle = run_length_encode(dna_bwt)
print("\nEncoded BWT:", dna_rle)
print(f"Length: {len(dna_bwt)} -> {len(dna_rle)}")

run_length_encode('aaabbc')  : 3a2bc | expected: 3a2bc
run_length_encode('annb$aa') : a2nb$2a | expected: a2nb$2a
run_length_encode('')        : '' | expected: ''

Encoded BWT: 2C2GTACGACGATGTGAGATCGCAG2CATCGA2CTGA3CGC3ATCTCTAG2TACAGT2AGT2A2TGT2G3ACTGAG$TCA3GCTCGA2CTATACGATAT2G2ACG3TCG2TG3TGAGCGTC3TGAGAG2CGAGACG2AGTC3A
Length: 151 -> 143


***

## Run-Length Decoding

Run-length decoding reverses the RLE process by expanding each stored pair back into its original sequence of repeated characters. When combined with the inverse BWT, this allows us to reconstruct the original text from a compressed, run-length encoded representation of the transform.

The pseudocode for decoding is straightforward:

$$
\small
\begin{array}{l}
\textbf{function } \text{run\_length\_decode}(\text{encoded text}) \\
\quad \text{Start with an empty decoded text} \\
\quad \text{Begin reading at the start of the encoded text} \\
\quad \textbf{while } \text{there is still unread encoded text} \textbf{ do} \\
\quad\quad \text{Read a single character, which is the symbol to be repeated} \\
\quad\quad \text{Then read the following one or more digits as the number of repeats} \\
\quad\quad \text{Turn this sequence of digits into a number} \\
\quad\quad \text{Add that many copies of the symbol to the decoded text} \\
\quad \textbf{return } \text{the decoded text}
\end{array}
$$


In [11]:
def run_length_decode(encoded: str) -> str:
    """Decode a run-length encoded character-count string.

    Reconstructs the original string by parsing an encoded representation
    where each run is stored as a character followed by its count, and
    expanding each run back into repeated characters.

    Args:
    encoded: A run-length encoded string of the form 'a3n2b1$1a2'.

    Returns:
    The decoded string obtained by expanding all runs in the encoded input.

    Examples:
    >>> run_length_decode('a3b2c1')
    'aaabbc'

    >>> run_length_decode('a1n2b1$1a2')
    'annb$aa'
    """
    # empty output, and a place to hold digits we haven't used yet
    decoded = ""
    digits = ""

    # go through the encoded string one character at a time
    for char in encoded:
        if char.isdigit():
            # part of a count, but more digits might follow, so hold onto it
            digits += char
        else:
            # a letter: the count in front of it (if any) is now complete
            if digits == "":
                repeat = 1
            else:
                repeat = int(digits)
            decoded += char * repeat
            # clear the digits so the next letter starts fresh
            digits = ""

    # return the rebuilt string
    return decoded

# Docstring examples
print("run_length_decode('3a2bc')   :", run_length_decode('3a2bc'), "| expected: aaabbc")
print("run_length_decode('a2nb$2a') :", run_length_decode('a2nb$2a'), "| expected: annb$aa")
print("run_length_decode('12a')     :", run_length_decode('12a'), "| expected: 12 a's")

# Testing the run_length_decode function with the generated DNA sequence
dna_decoded = run_length_decode(dna_rle)
print("\nRound trip matches original BWT:", dna_decoded == dna_bwt)



run_length_decode('3a2bc')   : aaabbc | expected: aaabbc
run_length_decode('a2nb$2a') : annb$aa | expected: annb$aa
run_length_decode('12a')     : aaaaaaaaaaaa | expected: 12 a's

Round trip matches original BWT: True


In [12]:
def main(reference: str, queries: list[str]) -> None:
    """Search a reference for patterns and compress its BWT.

    Args:
        reference: The DNA sequence to search in.
        queries: The patterns to look for.
    """
    print(f"Reference ({len(reference)} bp):\n{reference}\n")

    # 1. pattern search (builds the suffix array, BWT, count, and occur inside)
    print("Pattern search:")
    for q in queries:
        print(f"  {q:>10}: {find_match(q, reference)}")

    # 2. compress the BWT and check the round trip
    text = reference + '$'
    bwt = BWT_from_suffix_array(text, suffix_array(text))
    encoded = run_length_encode(bwt)
    print(f"\nRun-length encoded BWT ({len(bwt)} -> {len(encoded)} chars):\n{encoded}")
    print("Decodes back to the BWT:", run_length_decode(encoded) == bwt)


if __name__ == "__main__":
    reference = generate_dna(150, seed=42)
    queries = [reference[10:20], "GAC", "CATT", "TTTT"]
    main(reference, queries)

Reference (150 bp):
GACAGGTACAAGAAGGAGTATGCATCAATGTGGTCGTGTGGAACAAACGCCACTGGAGACTGGGTTAACCATTCGCTCCAGCGTCATGAAAGTCACTGTTAGGGCGACCTTCGATTCGGATGTGACATTTCATTACATTACGCTCAGGAC

Pattern search:
  AGAAGGAGTA: [10]
         GAC: [0, 57, 105, 123, 147]
        CATT: [69, 125, 130, 135]
        TTTT: []

Run-length encoded BWT (151 -> 143 chars):
2C2GTACGACGATGTGAGATCGCAG2CATCGA2CTGA3CGC3ATCTCTAG2TACAGT2AGT2A2TGT2G3ACTGAG$TCA3GCTCGA2CTATACGATAT2G2ACG3TCG2TG3TGAGCGTC3TGAGAG2CGAGACG2AGTC3A
Decodes back to the BWT: True


## Tricky Parts to Watch Out For

1. **End-of-string character**: Ensure your implementation includes a unique end-of-string character (often '$') that doesn't appear elsewhere in the string. This character should typically be lexicographically smaller than all other characters.

2. **Indexing issues**: Pay careful attention to zero-based versus one-based indexing, especially when implementing the BWT from suffix arrays.

3. **Edge cases**: Handle edge cases properly, such as when the suffix array index is 0, or when dealing with empty strings or patterns.

4. **Performance considerations**: The naive BWT implementation is $O(n^2 \log n)$ due to sorting n strings of length n. Using suffix arrays reduces this to $O(n \log n)$.

5. **Memory usage**: For large strings, storing all rotations can be memory-intensive. The suffix array approach is more memory-efficient.

---

# Summary

The Burrows-Wheeler Transform showcases how a seemingly simple reordering of characters can lead to powerful applications in compression and pattern matching. By understanding both the conceptual foundation and the implementation details, you'll gain insight into one of the most elegant algorithms in computer science, with applications ranging from everyday file compression to cutting-edge genomic research.